<h3> Tłumaczenie

NAAAAAAAAAAAAAAAAAJNOWZZE PROMPT: https://chatgpt.com/c/6a0090d9-03a0-838b-b388-ad528d96b5a6

https://chatgpt.com/c/69fb7ad7-f0d8-8331-906e-4f123a3e381f

moze jako projekt? https://github.com/kahotsang/image-captioning

https://chatgpt.com/c/69fcc2a0-a8e0-8327-9712-fe554502c738

skąd pobrac dane: https://github.com/Avaneesh40585/Flickr8k-Dataset

A moze flicker do domu a na zajęcih pokazac CIFAR? dostosowac kod


modele ypu CLIP pod spodem

moze stąd tez? https://www.kaggle.com/datasets/adityajn105/flickr8k

In [7]:
import os
import random
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

import torchvision.datasets as datasets

from transformers import BertTokenizer, BertModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
transform = transforms.Compose([
    transforms.ToTensor()
])
dataset = datasets.CIFAR10(root="cifar", train=True, download=False, transform=transform)

In [8]:

# ===== TEXT ENCODER (BERT) =====
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(device)

Dla uproszczenia weźmiemy obrazki z kategorii, w ogólności nie musimy ograniczać się do skończonej liczby

In [9]:
labels_map = {
    0: "airplane", 1: "automobile", 2: "bird", 3: "cat", 4: "deer",
    5: "dog", 6: "frog", 7: "horse", 8: "ship", 9: "truck"
}

class CIFARTextDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        text = f"a photo of a {labels_map[label]}"
        return img, text

dataset = CIFARTextDataset(dataset)


import numpy as np
from torch.utils.data import Subset

subset_size = int(0.1 * len(dataset))
indices = np.random.choice(len(dataset), subset_size, replace=False)
dataset_small = Subset(dataset, indices)
loader = DataLoader(dataset_small, batch_size=64, shuffle=True)
#loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [10]:
class TextEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.bert = bert
        self.fc = nn.Linear(768, embed_dim)

    def forward(self, texts):
        tokens = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = self.bert(**tokens)
        cls = outputs.last_hidden_state[:, 0, :]  # CLS token
        return F.normalize(self.fc(cls), dim=-1)

# ===== IMAGE ENCODER =====
class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2),
            nn.ReLU(),
        )

        # 🔥 automatyczne liczenie
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 32, 32)
            out = self.conv(dummy)
            self.flatten_dim = out.view(1, -1).shape[1]

        self.fc = nn.Linear(self.flatten_dim, embed_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return F.normalize(self.fc(x), dim=-1)

        
# ===== MODEL =====
text_encoder = TextEncoder().to(device)
image_encoder = ImageEncoder().to(device)

optimizer = torch.optim.Adam(
    list(text_encoder.parameters()) + list(image_encoder.parameters()),
    lr=1e-4
)

# ===== LOSS =====
def contrastive_loss(img_emb, txt_emb, temp=0.07):
    logits = img_emb @ txt_emb.T / temp
    labels = torch.arange(len(logits)).to(device)

    loss_i = F.cross_entropy(logits, labels)
    loss_t = F.cross_entropy(logits.T, labels)

    return (loss_i + loss_t) / 2

# ===== TRAIN =====
for epoch in range(3):
    k = 0
    for imgs, texts in loader:
        imgs = imgs.to(device)

        img_emb = image_encoder(imgs)
        txt_emb = text_encoder(texts)

        loss = contrastive_loss(img_emb, txt_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        k +=1
        print(k)

    print(f"Epoch {epoch}, loss: {loss.item():.4f}")

# ===== BUILD IMAGE DB =====
image_db = []
image_embs = []

for i in range(1000):  # subset
    img, text = dataset[i]
    img = img.unsqueeze(0).to(device)
    emb = image_encoder(img)
    image_db.append(img.cpu())
    image_embs.append(emb.detach())

image_embs = torch.cat(image_embs)

# ===== QUERY =====
def retrieve(query):
    with torch.no_grad():
        txt_emb = text_encoder([query])
        sims = txt_emb @ image_embs.T
        idx = sims.argmax().item()
        return image_db[idx]


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
Epoch 0, loss: 1.4007
1
2
3
4
5
6
7
8


KeyboardInterrupt: 

In [ ]:
# ===== DEMO =====
import matplotlib.pyplot as plt

query = "a photo of a dog"
img = retrieve(query)

plt.imshow(img.squeeze().permute(1,2,0))
plt.title(query)
plt.axis("off")
plt.show()

In [ ]:
# ===== DEMO =====
import matplotlib.pyplot as plt

query = "a small animal that people like to live with"
img = retrieve(query)

plt.imshow(img.squeeze().permute(1,2,0))
plt.title(query)
plt.axis("off")
plt.show()

In [ ]:
# ===== DEMO =====
import matplotlib.pyplot as plt

query = "dog and cat"
img = retrieve(query)

plt.imshow(img.squeeze().permute(1,2,0))
plt.title(query)
plt.axis("off")
plt.show()

https://chatgpt.com/c/69fcc2a0-a8e0-8327-9712-fe554502c738